# Neurodegenerative Disease Knowledge Graph — Build Notebook

This notebook walks through the whole pipeline step by step:
1. Install what we need
2. Pull real abstracts from PubMed
3. Extract subject-verb-object relationships with spaCy
4. Build and draw a knowledge graph with NetworkX

Run each cell in order (Shift+Enter). This is meant to be read and modified, not just run — change the search term, try different diseases, and see what the graph looks like.

## Step 1 — Install packages
Colab already has most data science packages, but not these specific ones. This only needs to run once per session.

In [ ]:
!pip install biopython spacy networkx -q
!python -m spacy download en_core_web_sm -q

## Step 2 — Pull abstracts from PubMed with Biopython's Entrez module

NCBI (who runs PubMed) asks that you identify yourself with an email address when you use their API — this isn't a login, just a courtesy so they can contact you if your usage causes problems. Put your real email in below.

**This cell needs internet access and won't run outside Colab/a normal machine** — it makes a live call to PubMed's servers.

In [ ]:
from Bio import Entrez

Entrez.email = "your_email@example.com"  # <-- put your real email here

def fetch_abstracts(query, max_results=15):
    """Search PubMed and return a list of abstract texts for the query."""
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
    record = Entrez.read(handle)
    handle.close()
    ids = record["IdList"]

    if not ids:
        return []

    handle = Entrez.efetch(db="pubmed", id=ids, rettype="abstract", retmode="text")
    raw = handle.read()
    handle.close()

    # PubMed returns abstracts separated by blank lines -- this is a simple
    # split, not perfect, but good enough for a first version of the project.
    abstracts = [a.strip() for a in raw.split("\n\n") if len(a.strip()) > 200]
    return abstracts

# Try it -- change this search term to whatever neurodegenerative disease
# angle you want to focus on for your Emory essay direction
abstracts = fetch_abstracts("Alzheimer's disease amyloid microglia", max_results=15)
print(f"Pulled {len(abstracts)} abstracts")
print(abstracts[0][:500] if abstracts else "No results")

## Step 3 — Extract subject-verb-object triples with spaCy

This is the core NLP step. spaCy's dependency parser identifies the grammatical structure of each sentence, and we pull out patterns like `(subject, verb, object)` — e.g. `("amyloid plaques", "disrupt", "neuronal function")`.

**Honest caveat to know for interviews/essays:** this is a beginner-friendly heuristic, not a state-of-the-art relation-extraction model. It will miss some sentences and occasionally extract a weird triple. That's normal, and knowing *why* it misses things (complex sentence structure, passive voice, multi-clause sentences) is actually a good thing to be able to explain if anyone asks about the project.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

def get_span_for_token(token):
    """Expand a single token to its full noun phrase, e.g. 'brain' -> 'the brain'."""
    for chunk in token.doc.noun_chunks:
        if token.i >= chunk.start and token.i < chunk.end:
            return chunk.text
    return token.text

def extract_triples(text, nlp):
    doc = nlp(text)
    triples = []
    for sent in doc.sents:
        subj, verb, obj = None, None, None
        for token in sent:
            if token.dep_ in ("nsubj", "nsubjpass") and token.head.pos_ == "VERB":
                subj = token
                verb = token.head
            if token.dep_ in ("dobj", "pobj", "attr") and verb is not None:
                if token.head == verb or token.head.head == verb:
                    obj = token
        if subj is not None and verb is not None and obj is not None:
            triples.append((get_span_for_token(subj), verb.lemma_, get_span_for_token(obj)))
    return triples

# Run it across every abstract we pulled
all_triples = []
for abstract in abstracts:
    all_triples.extend(extract_triples(abstract, nlp))

print(f"Extracted {len(all_triples)} triples")
for t in all_triples[:15]:
    print(" ", t)

## Step 4 — Build and draw the knowledge graph with NetworkX

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
for subj, verb, obj in all_triples:
    G.add_edge(subj, obj, label=verb)

print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, seed=42, k=1.2)
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=1800, font_size=7, arrowsize=15)
edge_labels = nx.get_edge_attributes(G, "label")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6)
plt.title("Neurodegenerative Disease Knowledge Graph")
plt.show()

## Next steps

- **If the graph looks sparse or messy:** try a more specific search term, pull more abstracts, or read up on spaCy's `noun_chunks` and dependency labels to refine the extraction rules above.
- **To build a stronger NLP foundation before going further:** work through Kaggle's short *"Intro to NLP"* micro-course and fast.ai's *"Getting started with NLP for absolute beginners"* notebook (search Kaggle for these by name — both are free, no login-walled content, and widely used).
- **Once this works reliably here:** the `app.py` file (Streamlit version of this exact pipeline) turns it into a clickable web app instead of a notebook. See the accompanying step-by-step guide for how to run and deploy it.